In [1]:
!pip install -q sentence-transformers pandas pyarrow tqdm

In [ ]:
%%writefile dataset_embedding_processor.py
#  ←  paste the entire script here

#!/usr/bin/env python3
"""
Dataset Embedding Processor – Colab‑ready
========================================

Updated **23 Apr 2025** to reflect the latest user requirements.

Key updates
-----------
* **Excludes** the columns `category`, `category`, `bucket_start_time`, and
  `time_bucket` when constructing the natural‑language description that gets
  encoded. These columns remain in the final data frame but do **not** influence
  the embedding vectors.
* The generated sentence is always saved as `text_representation`; pass
  `--drop-text` if you explicitly want it removed before writing the Parquet
  file.
* All other behavioural details (streaming, batching, optional `.npy` dumps)
  remain unchanged.

Run in Colab
------------
```bash
!pip install -q sentence-transformers pandas pyarrow tqdm

%%writefile dataset_embedding_processor.py
# ← paste this file (or clone from your repo)

!python dataset_embedding_processor.py /path/to/temp_dataset_features2.csv \
       -o /path/to/processed_chunks
```
"""
from __future__ import annotations

import argparse
import logging
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ─── Configuration Defaults ────────────────────────────────────────────────

DEFAULT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # fast, 384‑dim
DEFAULT_BATCH = 256  # encoder batch size; adjust for GPU RAM
DEFAULT_CHUNK = 250_000  # rows per streaming chunk
TEXT_COL_LIMIT = 256  # max tokens fed to the model

# Columns we do *not* want to embed
EXCLUDE_COLS = {
    "category",
    "category",
    "bucket_start_time",
    "time_bucket",
}

# ─── CLI ────────────────────────────────────────────────────────────────────

import sys

def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    """Parse CLI *or* notebook cell arguments.

    Colab / Jupyter adds its own flags (e.g. `-f <connection_file>`). We
    therefore use *parse_known_args* and silently ignore anything we don't
    recognise, to prevent `SystemExit: 2` errors when the script is executed in
    a notebook cell with `%run` or similar magic.
    """
    if argv is None:
        argv = sys.argv[1:]

    p = argparse.ArgumentParser(
        prog="dataset_embedding_processor.py",
        description="Stream a large CSV and add Sentence‑BERT embeddings row‑wise.",
    )
    p.add_argument("csv", type=Path, help="Input CSV file path.")
    p.add_argument(
        "-o",
        "--out",
        type=Path,
        default=Path("./processed_chunks"),
        help="Directory where Parquet chunks will be written.",
    )
    p.add_argument(
        "-m",
        "--model",
        default=DEFAULT_MODEL,
        help="HuggingFace model name or local path compatible with SentenceTransformer.",
    )
    p.add_argument("-b", "--batch", type=int, default=DEFAULT_BATCH, help="Embedding batch size.")
    p.add_argument("-c", "--chunk", type=int, default=DEFAULT_CHUNK, help="CSV chunk size.")
    p.add_argument(
        "--drop-text",
        action="store_true",
        help="Drop the text_representation column before writing parquet.",
    )
    p.add_argument("--to-npy", action="store_true", help="Also dump raw .npy embedding blobs.")

    args, unknown = p.parse_known_args(argv)
    if unknown:
        logging.warning("Ignoring unknown CLI args: %s", unknown)
    return args

# ─── Model Helper ────────────────────────────────────────────────────────── ──────────────────────────────────────────────────────────

def load_model(model_name: str) -> SentenceTransformer:
    logging.info("Loading Sentence‑BERT model %s", model_name)
    model = SentenceTransformer(model_name)
    model.max_seq_length = TEXT_COL_LIMIT  # safety clip
    return model

# ─── Row → Text ────────────────────────────────────────────────────────────

def row_to_text(row: pd.Series) -> str:
    """Turn a dataframe row into a compact natural‑language sentence.

    * Skips columns in **EXCLUDE_COLS**.
    * Omits numeric values equal to 0 (common in this dataset).
    """
    parts: List[str] = []

    pretty_map = {
        "avg_read_throughput": "average read throughput",
        "avg_write_throughput": "average write throughput",
        "read_LBA_variance": "read LBA variance",
        "write_LBA_variance": "write LBA variance",
        "write_entropy_mean": "write entropy mean",
        "mem_read_entropy_mean": "memory read entropy mean",
        "mem_write_entropy_mean": "memory write entropy mean",
        "mem_readwrite_entropy_mean": "memory read‑write entropy mean",
        "mem_read_GPA_variance": "memory read GPA variance",
        "mem_write_GPA_variance": "memory write GPA variance",
        "mem_readwrite_GPA_variance": "memory read‑write GPA variance",
        "mem_exec_GPA_variance": "memory exec GPA variance",
    }

    for col, val in row.items():
        if col in EXCLUDE_COLS or pd.isna(val):
            continue
        if isinstance(val, (int, float)) and val == 0:
            continue  # skip zero‑only metrics for brevity

        category" = pretty_map.get(col, col.replace("_", " "))
        parts.append(f"{category} {val}")

    return "; ".join(parts[:50])  # cap to avoid overly long sentences

# ─── Embedding Helper ──────────────────────────────────────────────────────

def embed_texts(model: SentenceTransformer, texts: List[str], batch_size: int) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        device=model.device,
    ).astype("float32")

# ─── Main Processor ────────────────────────────────────────────────────────

def process_csv(
    csv_file: Path,
    out_dir: Path,
    model: SentenceTransformer,
    batch: int,
    chunk_size: int,
    drop_text: bool = False,
    dump_npy: bool = False,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    for idx, chunk in enumerate(
        tqdm(pd.read_csv(csv_file, chunksize=chunk_size), desc="Reading CSV", unit="chunk")
    ):
        # Create NLP representation
        chunk["text_representation"] = chunk.apply(row_to_text, axis=1)

        # Embed
        embeddings = embed_texts(model, chunk["text_representation"].tolist(), batch)
        chunk["embedding"] = embeddings.tolist()

        # Optionally drop the text column
        if drop_text:
            chunk = chunk.drop(columns="text_representation")

        # Save chunk
        parquet_path = out_dir / f"chunk_{idx:03d}.parquet"
        chunk.to_parquet(parquet_path, index=False)
        logging.info("Saved %s (%d rows)", parquet_path, len(chunk))

        # Optional .npy dump
        if dump_npy:
            npy_path = out_dir / f"chunk_{idx:03d}_embeddings.npy"
            np.save(npy_path, embeddings)
            logging.info("Saved %s", npy_path)

# ─── Entry Point ───────────────────────────────────────────────────────────

def main() -> None:
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

    model = load_model(args.model)

    process_csv(
        csv_file=args.csv,
        out_dir=args.out,
        model=model,
        batch=args.batch,
        chunk_size=args.chunk,
        drop_text=args.drop_text,
        dump_npy=args.to_npy,
    )


if __name__ == "__main__":
    main()


Writing dataset_embedding_processor.py


In [4]:
!python dataset_embedding_processor.py \
    /mnt/4TBSTORE/Dr_Kabuye4TB/RanSMAP/temp_dataset_features2.csv \
    -o /content/processed_chunks

2025-04-23 12:53:03.770098: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745409183.835446   10130 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745409183.855263   10130 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-23 12:53:03.991450: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/home/henrykabuye/anaconda3/envs/kabuye1_env/lib/python3.9/site-packages/transform

In [ ]:
# !pip install -q sentence-transformers pyarrow pandas shap pyswarms ctgan

# # --------------------------------------------
# # 1.  Load + massage the parquet file
# # --------------------------------------------



import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
import pyswarms as ps



DATA_PATH = "/mnt/4TBSTORE/Dr_Kabuye4TB/RanSMAP/final_embeddings.parquet"
df = pd.read_parquet(DATA_PATH)

# 1️⃣  Drop the verbose text-column if it’s present
df = df.drop(columns=[c for c in df.columns if c == "text_representation"], errors="ignore")

# 2️⃣  Expand the list-of-floats in the “embedding” column → 384 numeric cols
#     (If you used a different model, adjust the 384.)
if "embedding" not in df.columns:
    raise ValueError("⚠️  No 'embedding' column found. Did you load the right file?")

emb_df = pd.DataFrame(df["embedding"].tolist()).add_prefix("emb_")
df = pd.concat([df.drop(columns="embedding"), emb_df], axis=1)

# --------------------------------------------
# 2.  Identifiers for *semantic* vs *behavioral* inputs
# --------------------------------------------
semantic_cols   = [c for c in df.columns if c.startswith("emb_")]      # ← 384-dim
behavioral_cols = [
    c for c in df.columns
    if c not in semantic_cols + ["category", "label"]                  # keep numeric / entropy etc.
]

print(f"✅ semantic  (embedding) dims: {len(semantic_cols)}")
print(f"✅ behavioral feature dims  : {len(behavioral_cols)}")
print()

# # --------------------------------------------
# # 3.  Optional: CTGAN zero-variance imputation
# # --------------------------------------------
# from ctgan import CTGAN
# import numpy as np

# zero_cols = [c for c in behavioral_cols if df[c].nunique() <= 1]
# print("🩻 zero-variance cols:", zero_cols)

# if zero_cols:
#     non_zero_df = df.drop(columns=zero_cols).copy()

#     # categorical / discrete variables for CTGAN
#     discrete_cols = [
#         c for c in non_zero_df.columns
#         if (non_zero_df[c].dtype == "object") or (c in ["category", "label"])
#     ]
#     # ➡︎ cast *only* those to str
#     for c in discrete_cols:
#         non_zero_df[c] = non_zero_df[c].astype(str)

#     ctgan = CTGAN(epochs=5, verbose=False)
#     ctgan.fit(non_zero_df, discrete_columns=discrete_cols)  # <-- no global .astype(str)!
#     synth = ctgan.sample(len(df))

#     # replace zeros only where they occur
#     for col in zero_cols:
#         if col in synth.columns:
#             df[col] = np.where(df[col] == 0, synth[col], df[col])

#     print("✅ zero-variance columns replaced with CTGAN samples.")
# else:
#     print("✅ No zero-variance columns found – CTGAN step skipped.")


# -------------------------------
# Semantic Embedding PCA Compression
# -------------------------------
from sklearn.decomposition import PCA

TARGET_SEMANTIC_DIM = 128  # or 64 if you want smaller

print("🎯 Compressing semantic embeddings...")

pca = PCA(n_components=TARGET_SEMANTIC_DIM, random_state=42)
X_sem_pca = pca.fit_transform(X_sem)

print(f"✅ Semantic dims reduced: {X_sem.shape[1]} → {X_sem_pca.shape[1]}")


########

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🚀 using device:", device)

X_sem = df[semantic_cols].values.astype(np.float32)
X_beh = df[behavioral_cols].values.astype(np.float32)
y     = df["category"].astype(np.float32).values          # ⇐ binary labels assumed

scaler   = StandardScaler()
X_beh    = scaler.fit_transform(X_beh)

# Redefine splits
X_sem_tr, X_sem_te, X_beh_tr, X_beh_te, y_tr, y_te = train_test_split(
    X_sem_pca, X_beh, y, test_size=0.20, random_state=42, stratify=y
)

train_ds = TensorDataset(
    to_tensor(X_sem_tr), to_tensor(X_beh_tr), to_tensor(y_tr)
)
test_ds = TensorDataset(
    to_tensor(X_sem_te), to_tensor(X_beh_te), to_tensor(y_te)
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

print("✅ data ready →", len(train_ds), "train rows &", len(test_ds), "test rows")
print("✅ Retraining loaders built.")








# # --------------------------------------------
# # 4.  Train/test splits → Torch tensors
# # --------------------------------------------
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("🚀 using device:", device)

# X_sem = df[semantic_cols].values.astype(np.float32)
# X_beh = df[behavioral_cols].values.astype(np.float32)
# y     = df["category"].astype(np.float32).values          # ⇐ binary labels assumed

# scaler   = StandardScaler()
# X_beh    = scaler.fit_transform(X_beh)

# X_sem_tr, X_sem_te, X_beh_tr, X_beh_te, y_tr, y_te = train_test_split(
#     X_sem, X_beh, y, test_size=0.20, random_state=42, stratify=y
# )

# def to_tensor(x): return torch.tensor(x, device=device)

# train_ds = TensorDataset(
#     to_tensor(X_sem_tr), to_tensor(X_beh_tr), to_tensor(y_tr)
# )
# test_ds  = TensorDataset(
#     to_tensor(X_sem_te), to_tensor(X_beh_te), to_tensor(y_te)
# )
# train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
# test_loader  = DataLoader(test_ds,  batch_size=256)

# print("✅ data ready →", len(train_ds), "train rows &", len(test_ds), "test rows")

## -------------------------------
# 4. Model Definitions (CNN version)
# -------------------------------

class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        return self.model(x)

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        return self.model(x)

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, sem_out, beh_out):
        combined = torch.cat([sem_out, beh_out], dim=1)
        return self.model(combined).squeeze()

In [ ]:
# -------------------------------
# 9. Full PSO + Retraining Pipeline
# -------------------------------

import time
import matplotlib.pyplot as plt

X_sem_train, X_sem_test, X_beh_train, X_beh_test, y_train, y_test = train_test_split(
    X_sem, X_beh, y, test_size=0.20, random_state=42, stratify=y
)

# -------------------------------
# PSO Objective Function
# -------------------------------
def pso_objective(params):
    dropout_rate = params[:, 0]
    lr = params[:, 1]
    losses = []

    for i in range(params.shape[0]):
        sem_agent = CNNSemanticAgent(X_sem_train.shape[1], dropout_rate[i]).to(device)
        beh_agent = CNNBehavioralAgent(X_beh_train.shape[1], dropout_rate[i]).to(device)
        decision_agent = DecisionAgent(32, 32, dropout_rate[i]).to(device)

        optimizer = torch.optim.Adam(
            list(sem_agent.parameters()) + list(beh_agent.parameters()) + list(decision_agent.parameters()),
            lr=lr[i]
        )
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)
        loss_fn = nn.BCELoss()

        train_losses, val_accuracies = [], []

        sem_agent.train(); beh_agent.train(); decision_agent.train()
        for epoch in range(5):  # quick inner training
            for xb_sem, xb_beh, yb in train_loader:
                sem_out = sem_agent(xb_sem)
                beh_out = beh_agent(xb_beh)
                preds = decision_agent(sem_out, beh_out)
                loss = loss_fn(preds, yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            scheduler.step()

            # Quick validation
            sem_agent.eval(); beh_agent.eval(); decision_agent.eval()
            val_preds, val_targets = [], []
            with torch.no_grad():
                for xb_sem, xb_beh, yb in test_loader:
                    sem_out = sem_agent(xb_sem)
                    beh_out = beh_agent(xb_beh)
                    probs = decision_agent(sem_out, beh_out)
                    val_preds.extend((probs > 0.5).int().cpu().numpy())
                    val_targets.extend(yb.cpu().numpy())
            acc = accuracy_score(val_targets, val_preds)
            val_accuracies.append(acc)

        losses.append(1 - np.mean(val_accuracies))  # want to minimize 1-accuracy

    return np.array(losses)

# -------------------------------
# Run PSO Optimization
# -------------------------------
print("🚀 Running PSO to optimize dropout and learning rate...")

from pyswarms.single import GlobalBestPSO

bounds = ([0.1, 1e-4], [0.5, 1e-2])
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}

start_time = time.time()

optimizer = GlobalBestPSO(
    n_particles=5,
    dimensions=2,
    options=options,
    bounds=bounds
)

best_cost, best_params = optimizer.optimize(pso_objective, iters=5)

end_time = time.time()

print(f"✅ PSO Done in {end_time - start_time:.1f} seconds")
print(f"✅ Best Dropout: {best_params[0]:.4f}")
print(f"✅ Best Learning Rate: {best_params[1]:.6f}")

# -------------------------------
# Retrain CNN using Best Hyperparameters
# -------------------------------
print("🔄 Retraining CNN fusion model with best hyperparameters...")

best_dropout = best_params[0]
best_lr = best_params[1]

sem_agent = CNNSemanticAgent(X_sem_train.shape[1], best_dropout).to(device)
beh_agent = CNNBehavioralAgent(X_beh_train.shape[1], best_dropout).to(device)
decision_agent = DecisionAgent(32, 32, best_dropout).to(device)

optimizer = torch.optim.Adam(
    list(sem_agent.parameters()) + list(beh_agent.parameters()) + list(decision_agent.parameters()),
    lr=best_lr
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)
loss_fn = nn.BCELoss()

train_losses = []
val_accuracies = []

EPOCHS = 15  # More epochs since it's the final retrain

for epoch in range(EPOCHS):
    sem_agent.train(); beh_agent.train(); decision_agent.train()
    total_loss = 0
    for xb_sem, xb_beh, yb in train_loader:
        sem_out = sem_agent(xb_sem)
        beh_out = beh_agent(xb_beh)
        preds = decision_agent(sem_out, beh_out)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    sem_agent.eval(); beh_agent.eval(); decision_agent.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for xb_sem, xb_beh, yb in test_loader:
            sem_out = sem_agent(xb_sem)
            beh_out = beh_agent(xb_beh)
            probs = decision_agent(sem_out, beh_out)
            val_preds.extend((probs > 0.5).int().cpu().numpy())
            val_targets.extend(yb.cpu().numpy())
    acc = accuracy_score(val_targets, val_preds)
    train_losses.append(total_loss)
    val_accuracies.append(acc)
    print(f"📦 Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss:.4f} | Val Acc: {acc:.4f}")

# -------------------------------
# Plot the Training Results
# -------------------------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.title("Training Loss over Epochs")
plt.legend()

plt.subplot(1,2,2)
plt.plot(val_accuracies, label="Validation Accuracy", color="green")
plt.title("Validation Accuracy over Epochs")
plt.legend()

plt.show()

In [ ]:
# 📊 Ransomware Detection Pipeline (PCA + PSO + CNN + MCTS)

# ✅ Step-by-step automatic pipeline:
# - PCA compression tuning
# - PSO hyperparameter optimization
# - CNN fusion training
# - MCTS evaluation with timing
# - Model saving

# ---------------------------------------------
# 📦 1. Setup & Install Dependencies
# ---------------------------------------------
# !pip install -q sentence-transformers pyarrow shap pyswarms ctgan

import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import pyswarms as ps
from imblearn.over_sampling import SMOTE
from ctgan import CTGAN
import time

# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# ---------------------------------------------
# 📥 2. Load Data
# ---------------------------------------------
DATA_PATH = "/content/drive/MyDrive/RanSMAP/final_embeddings.parquet"
df = pd.read_parquet(DATA_PATH)
df = df.drop(columns=[c for c in df.columns if c == "text_representation"], errors="ignore")

# Expand embedding
emb_df = pd.DataFrame(df["embedding"].tolist()).add_prefix("emb_")
df = pd.concat([df.drop(columns="embedding"), emb_df], axis=1)

semantic_cols = [c for c in df.columns if c.startswith("emb_")]
behavioral_cols = [c for c in df.columns if c not in semantic_cols + ["category", "label"]]

X_sem = df[semantic_cols].values.astype(np.float32)
X_beh = df[behavioral_cols].values.astype(np.float32)
y = df["category"].astype(np.float32).values

scaler = StandardScaler()
X_beh = scaler.fit_transform(X_beh)

# ---------------------------------------------
# ⚖️ 3. Impute Zero-Variance Behavioral Features
# ---------------------------------------------

print("Before rebalancing:", np.bincount(y.astype(int)))

def rebalance_dataset(X_df, y_series, method="SMOTE"):
    print(f"🔁 Rebalancing with {method}...")

    if method.upper() == "SMOTE":
        sm = SMOTE(random_state=42)
        X_resampled, y_resampled = sm.fit_resample(X_df, y_series)
        print("✅ SMOTE rebalancing complete.")
        return pd.DataFrame(X_resampled, columns=X_df.columns), pd.Series(y_resampled)

    elif method.upper() == "CTGAN":
        data = pd.concat([X_df, y_series.rename("category")], axis=1)
        discrete_cols = ["category"]

        ctgan = CTGAN(epochs=10)
        ctgan.fit(data, discrete_columns=discrete_cols)
        synth_data = ctgan.sample(len(data))

        X_synth = synth_data.drop(columns=["category"])
        y_synth = synth_data["category"]
        print("✅ CTGAN rebalancing complete.")
        return X_synth, y_synth

    else:
        raise ValueError("Invalid method. Choose 'SMOTE' or 'CTGAN'.")
X_df = pd.DataFrame(np.hstack([X_sem, X_beh]))
# X_bal, y_bal = rebalance_dataset(X_df, pd.Series(y), method="SMOTE")

# Choose one method dynamically
USE_CTGAN = False
rebalance_method = "CTGAN" if USE_CTGAN else "SMOTE"
X_bal, y_bal = rebalance_dataset(X_df, pd.Series(y), method=rebalance_method)


print("After rebalancing:", np.bincount(y_bal.astype(int)))
# Split back into semantic + behavioral after rebalancing
X_sem = X_bal.iloc[:, :X_sem.shape[1]].values
X_beh = X_bal.iloc[:, X_sem.shape[1]:].values
y = y_bal.values

# ---------------------------------------------
# 🧪 4. Auto PCA Compression Tuning
# ---------------------------------------------
def to_tensor(x):
    return torch.tensor(x, device=device)

class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
    def forward(self, x): return self.model(x.unsqueeze(1))

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
    def forward(self, x): return self.model(x.unsqueeze(1))

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, s, b): return self.model(torch.cat([s, b], dim=1)).squeeze()

def try_pca_compressions(X_sem, X_beh, y, target_dims):
    results = {}
    for dim in target_dims:
        pca = PCA(n_components=dim, random_state=42)
        X_pca = pca.fit_transform(X_sem)
        X_st, X_se, X_bt, X_be, yt, ye = train_test_split(X_pca, X_beh, y, test_size=0.2, stratify=y)
        train = DataLoader(TensorDataset(to_tensor(X_st), to_tensor(X_bt), to_tensor(yt)), batch_size=256)
        test = DataLoader(TensorDataset(to_tensor(X_se), to_tensor(X_be), to_tensor(ye)), batch_size=256)
        model = CNNSemanticAgent(dim, 0.3).to(device)
        beh = CNNBehavioralAgent(X_bt.shape[1], 0.3).to(device)
        dec = DecisionAgent(32, 32, 0.3).to(device)
        opt = torch.optim.Adam(list(model.parameters()) + list(beh.parameters()) + list(dec.parameters()), lr=1e-3)
        for _ in range(5):
            for xs, xb, yb in train:
                sem = model(xs)
                b = beh(xb)
                pred = dec(sem, b)
                loss = nn.BCELoss()(pred, yb)
                opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            preds, targets = [], []
            for xs, xb, yb in test:
                p = dec(model(xs), beh(xb))
                preds.extend((p > 0.5).int().cpu().numpy())
                targets.extend(yb.cpu().numpy())
            results[dim] = accuracy_score(targets, preds)
    return results

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Run PCA Search
search_dims = [128, 64, 32]
pca_results = try_pca_compressions(X_sem, X_beh, y, search_dims)
best_dim = max(pca_results, key=pca_results.get)
print(f"\n🏆 Best semantic compression: {best_dim} → Val Acc: {pca_results[best_dim]:.4f}")

# Rebuild dataset with best PCA
X_sem = PCA(n_components=best_dim, random_state=42).fit_transform(X_sem)
X_sem_train, X_sem_test, X_beh_train, X_beh_test, y_train, y_test = train_test_split(
    X_sem, X_beh, y, test_size=0.2, stratify=y, random_state=42
)

train_loader = DataLoader(TensorDataset(to_tensor(X_sem_train), to_tensor(X_beh_train), to_tensor(y_train)), batch_size=256)
test_loader  = DataLoader(TensorDataset(to_tensor(X_sem_test), to_tensor(X_beh_test), to_tensor(y_test)), batch_size=256)

# ✅ Ready for PSO (continue from here with optimization + MCTS)
print("✅ Data ready with PCA-compressed semantic features. Proceed to PSO and MCTS.")


In [2]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load real dataset
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/RanSMAP/final_embeddings.parquet")
df = df.drop(columns=["text_representation"], errors="ignore")

# Expand 'embedding' column into individual features
emb_df = pd.DataFrame(df["embedding"].tolist()).add_prefix("emb_")
df = pd.concat([df.drop(columns="embedding"), emb_df], axis=1)

# Define feature groups
semantic_cols = [c for c in df.columns if c.startswith("emb_")]
behavioral_cols = [c for c in df.columns if c not in semantic_cols + ["category", "label"]]

# Extract features and labels
X_sem = df[semantic_cols].values.astype(np.float32)
X_beh = df[behavioral_cols].values.astype(np.float32)

benign_labels = ['Idle', 'Firefox', 'Office', 'AESCrypt', 'SDelete', 'Zip']
df['label'] = df['label'].apply(lambda x: 0 if x in BENIGN_LABELS else 1)
y = df["label"].astype(np.float32).values


# Optional: standardize behavioral features
X_beh = StandardScaler().fit_transform(X_beh)

# Apply SMOTE
from imblearn.over_sampling import SMOTE
X_combined = np.hstack([X_sem, X_beh])
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(X_combined, y)

# Split resampled features back
X_sem = X_resampled[:, :X_sem.shape[1]]
X_beh = X_resampled[:, X_sem.shape[1]:]
y = y_resampled

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_tensor(x): return torch.tensor(x, dtype=torch.float32, device=device)

# Models
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
    def forward(self, x): return self.model(x)

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
    def forward(self, x): return self.model(x)

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, s, b): return self.model(torch.cat([s, b], dim=1)).squeeze()

# Optuna objective
def objective(trial):
    max_dim = min(X_sem.shape[1], 128)
    pca_dim = trial.suggest_categorical("pca_dim", [d for d in [32, 64, 128] if d <= max_dim])
    drop_sem = trial.suggest_float("dropout_sem", 0.1, 0.5)
    drop_beh = trial.suggest_float("dropout_beh", 0.1, 0.5)
    drop_dec = trial.suggest_float("dropout_dec", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    # X_sem_pca = PCA(n_components=pca_dim).fit_transform(X_sem)
    # X_st, X_se, X_bt, X_be, yt, ye = train_test_split(X_sem_pca, X_beh, y, test_size=0.2, stratify=y)
    X_st_raw, X_se_raw, X_bt, X_be, yt, ye = train_test_split(X_sem, X_beh, y, test_size=0.2, stratify=y)
    pca = PCA(n_components=pca_dim).fit(X_st_raw)
    X_st = pca.transform(X_st_raw)
    X_se = pca.transform(X_se_raw)

    train_loader = DataLoader(TensorDataset(to_tensor(X_st), to_tensor(X_bt), to_tensor(yt)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_se), to_tensor(X_be), to_tensor(ye)), batch_size=256)

    sem = CNNSemanticAgent(pca_dim, drop_sem).to(device)
    beh = CNNBehavioralAgent(X_bt.shape[1], drop_beh).to(device)
    dec = DecisionAgent(32, 32, drop_dec).to(device)
    opt = torch.optim.Adam(list(sem.parameters()) + list(beh.parameters()) + list(dec.parameters()), lr=lr)

    for _ in range(5):
        sem.train(); beh.train(); dec.train()
        for xs, xb, yb in train_loader:
            xs = xs.unsqueeze(1) if xs.ndim == 2 else xs
            xb = xb.unsqueeze(1) if xb.ndim == 2 else xb
            pred = dec(sem(xs), beh(xb))
            loss = nn.BCELoss()(pred, yb)
            opt.zero_grad(); loss.backward(); opt.step()

    sem.eval(); beh.eval(); dec.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xs, xb, yb in test_loader:
            xs = xs.unsqueeze(1) if xs.ndim == 2 else xs
            xb = xb.unsqueeze(1) if xb.ndim == 2 else xb
            p = dec(sem(xs), beh(xb))
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())

    return accuracy_score(targets, preds)

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Collect embeddings for t-SNE visualization
semantic_features, behavioral_features, labels = [], [], []
with torch.no_grad():
    for xs, xb, yb in test_loader:
        xs = xs.unsqueeze(1) if xs.ndim == 2 else xs
        xb = xb.unsqueeze(1) if xb.ndim == 2 else xb
        sem_feat = sem(xs).cpu().numpy()
        beh_feat = beh(xb).cpu().numpy()
        semantic_features.append(sem_feat)
        behavioral_features.append(beh_feat)
        labels.append(yb.cpu().numpy())

# Stack and reduce
semantic_features = np.vstack(semantic_features)
behavioral_features = np.vstack(behavioral_features)
labels = np.concatenate(labels)

combined = np.hstack([semantic_features, behavioral_features])
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(combined)

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(X_tsne[labels == 0, 0], X_tsne[labels == 0, 1], c='blue', label='Benign', alpha=0.6)
plt.scatter(X_tsne[labels == 1, 0], X_tsne[labels == 1, 1], c='red', label='Ransomware', alpha=0.6)
plt.legend()
plt.title("t-SNE Projection of Learned Features")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.grid(True)
plt.tight_layout()
plt.show()


# Run optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
print("Best trial:", study.best_trial)

NameError: name 'test_loader' is not defined